# Multinomial Logistic Regression
## DS-3001: Foundations of Machine Learning

Content adapted from Terence Johnson (UVA)

### Loading Environment

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os # For changing directory

# To mount your google drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
path_to_DS_3001_folder = '/content/drive/MyDrive/DS-3001/03_Tuning_Testing_Evaluating'
# path_to_DS_3001_folder = ''

# Update the path to your folder for the class
# Where you stored the data from the previous noteboook
os.chdir(path_to_DS_3001_folder)

## Multinomial Logit

- **How do we expand logistic regression to handle multiple classes?**

- **Multiple Latent Variables:** Instead of calculating a single latent variable for the positive class, we can now calculate a latent variable, $L_{ik} = \beta_k \cdot x_i$, for each outcome $k=1,...,K$.

- **What do the latent variables represent?:** Imagine you, $i$, deciding which make/model of a car $k$ to buy, or which cereal $k$ to purchase: $L_{ik}$ is roughly how much you liked option $k$, which isn't observed. We only observe what you actually picked.

- **Identifying the Outcome from these Latent Variables:** The outcome is the maximum value over all the latent indices $(L_{i1}, L_{i2}, ..., L_{iK})$:
$$
y_{ik} = \begin{cases}
1, & \quad  L_{ik} > L_{ij}, j \neq k\\
0, & \quad \text{otherwise.}
\end{cases}
$$

- So for each observation $i$ and outcome $k$, we just observe whether $i$ chooses $k$ or not, and not the latent indices $L_{ik}$

- In multinomial logistic regression, we want to recover the underlying preferences, the $\beta_1, ..., \beta_K$ coefficients.

- We can write this up on the board to compare with binary logistic regression.

## Categorical Cross Entropy

- **How do we decide what these coefficients, $\beta_1, ..., \beta_K$, should be?**

- We can expand our cross entropy loss function to include multiple classes.

- **Loss function from before:** Remember we expressed our binary cross entropy as:

$$
\text{bce}(b) = - \frac{1}{N} \sum_{i=1}^N [y_i \log \left( \hat{p}_i(b) \right) + (1-y_i) \log \left( 1-\hat{p}_i(b) \right)]
$$

- **New loss function:** We can express the categorical cross entropy as:

$$
\text{cce}(b) = - \dfrac{1}{N} \sum_{i=1}^N \sum_{k=1}^{K} y_{ik} \log( \hat{p}_{ik})
$$

- **Same intuition as before:** We want to maximize the predicted probability of the true class. For example, if the true class for observation i=1 were k=2, we want to maximize $\hat{p}_{1,2}$.

- **Calculating the probabilities from our latent variables:** We can calculate the probability of each class, k, using the **softmax** function:

$$
\hat{p}_{ik}(y_i=k | x_i) = \text{Softmax}(L_k) = \frac{e^{L_k}}{\sum^{K}_{j=1}e^{L_j}}
$$

- The softmax ensures that probabilities sum to 1.

### Example: Land Mines

In [ ]:
import pandas as pd

# Load in the LogisticRegression function
from sklearn.linear_model import LogisticRegression

# Define a min max function
def MinMax(X):
  result = (X - X.min()) / (X.max() - X.min())
  return result

# Load in the data set
mine_df = pd.read_csv('./data/land_mines.csv')

# What features are in this data set?
print('Mine DF Head:', mine_df.head())

# Isolate the outcome and the design matrix, just using the raw featres as inputs for now
y_mines = mine_df['mine_type']
X_mines = mine_df.drop('mine_type',axis=1)

# Check if they're standardized
print('X_mines description before scaling:', X_mines.describe())

# Apply min max scaling
X_mines = MinMax(X_mines)
print('X_mines description after scaling:', X_mines.describe())

In [ ]:
# Fit a multinomial logistic regression model to the data
mnl = LogisticRegression(penalty=None).fit(X_mines,y_mines)

# Isolate out the Regression coefficients
mnl_coef = pd.DataFrame(mnl.coef_)

# Replace the columns names so that they're the feature names and not just indices
mnl_coef.columns = mnl.feature_names_in_

# Print them to the screen
print('Coefficients for each landmine type:')
mnl_coef

**Questions to consider:**

1. How many coefficients are there in total?

2. How many coefficients are there for each input variable?

2. How does this compare to binary logistic regression?

3. How could we write out this model for each latent variable?

In [ ]:
# We can calcaulate the predicted probabilities for each of class
pd.DataFrame(mnl.predict_proba(X_mines))

In [ ]:
# We can calculate the confusion matrix for our
tab = pd.crosstab(
    y_mines, # The true class values
    mnl.predict(X_mines) # The hard predictions for each observation
)

tab

In [ ]:
# We can calculate the accuracy for this particular model
acc = np.trace(tab)/len(y_mines)
print('Accuracy: ', acc)